In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS dev_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS dev_silver.accuweather")

In [0]:
%sql
CREATE TABLE dev_silver.accuweather.forecast_daily_calendar_imperial ( city_name STRING, country_code STRING, latitude DOUBLE, longitude DOUBLE, date DATE, cloud_cover_perc_avg INT, cloud_cover_perc_max INT, cloud_cover_perc_min INT, degree_days_cooling DOUBLE, degree_days_freezing DOUBLE, degree_days_growing DOUBLE, degree_days_heating DOUBLE, humidity_relative_avg DOUBLE, humidity_relative_max DOUBLE, humidity_relative_min DOUBLE, index_air_quality_24hr_max DOUBLE, index_uv_avg DOUBLE, index_uv_max DOUBLE, index_uv_min DOUBLE, minutes_of_ice_total INT, minutes_of_precipitation_total INT, minutes_of_sun_total INT, minutes_of_rain_total INT, minutes_of_snow_total INT, has_ice BOOLEAN, ice_lwe_rate_avg DOUBLE, ice_lwe_rate_max DOUBLE, ice_lwe_rate_min DOUBLE, ice_lwe_total DOUBLE, ice_probability INT, has_precipitation BOOLEAN, precipitation_lwe_rate_avg DOUBLE, precipitation_lwe_rate_max DOUBLE, precipitation_lwe_rate_min DOUBLE, precipitation_lwe_total DOUBLE, precipitation_probability INT, precipitation_type_desc_predominant STRING, has_rain BOOLEAN, rain_lwe_rate_avg DOUBLE, rain_lwe_rate_max DOUBLE, rain_lwe_rate_min DOUBLE, rain_lwe_total DOUBLE, rain_probability INT, snow_liquid_ratio_accuweather_avg DOUBLE, snow_liquid_ratio_accuweather_max DOUBLE, snow_liquid_ratio_accuweather_min DOUBLE, has_snow BOOLEAN, snow_avg DOUBLE, snow_max DOUBLE, snow_min DOUBLE, snow_total DOUBLE, snow_lwe_rate_avg DOUBLE, snow_lwe_rate_max DOUBLE, snow_lwe_rate_min DOUBLE, snow_lwe_total DOUBLE, snow_probability INT, solar_irradiance_avg DOUBLE, solar_irradiance_max DOUBLE, solar_irradiance_total DOUBLE, temperature_avg DOUBLE, temperature_max DOUBLE, temperature_min DOUBLE, temperature_dew_point_avg DOUBLE, temperature_dew_point_max DOUBLE, temperature_dew_point_min DOUBLE, temperature_heat_index_avg DOUBLE, temperature_heat_index_max DOUBLE, temperature_heat_index_min DOUBLE, temperature_realfeel_avg DOUBLE, temperature_realfeel_max DOUBLE, temperature_realfeel_min DOUBLE, temperature_realfeel_shade_avg DOUBLE, temperature_realfeel_shade_max DOUBLE, temperature_realfeel_shade_min DOUBLE, temperature_wind_chill_avg DOUBLE, temperature_wind_chill_max DOUBLE, temperature_wind_chill_min DOUBLE, visibility_avg DOUBLE, visibility_max DOUBLE, visibility_min DOUBLE, wind_direction_avg DOUBLE, wind_gust_avg DOUBLE, wind_gust_max DOUBLE, wind_gust_min DOUBLE, wind_gust_direction_avg DOUBLE, wind_speed_avg DOUBLE, wind_speed_max DOUBLE, wind_speed_min DOUBLE) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')

In [0]:
df = spark.table("dev_bronze.researcher_api.forecast_daily_calendar_imperial")

df_converted = df.select(
    "city_name",
    "country_code",
    "latitude",
    "longitude",
    "date",
    ((df.temperature_avg - 32) * 5/9).alias("temperature_avg_c"),
    ((df.temperature_max - 32) * 5/9).alias("temperature_max_c"),
    ((df.temperature_min - 32) * 5/9).alias("temperature_min_c"),
    (df.precipitation_lwe_total * 25.4).alias("precipitation_lwe_total_mm"),
    (df.precipitation_lwe_rate_avg * 25.4).alias("precipitation_lwe_rate_avg_mm"),
    (df.precipitation_lwe_rate_max * 25.4).alias("precipitation_lwe_rate_max_mm"),
    (df.precipitation_lwe_rate_min * 25.4).alias("precipitation_lwe_rate_min_mm"),
    (df.snow_total * 2.54).alias("snow_total_cm"),
    (df.snow_avg * 2.54).alias("snow_avg_cm"),
    (df.snow_max * 2.54).alias("snow_max_cm"),
    (df.snow_min * 2.54).alias("snow_min_cm"),
    (df.wind_speed_avg * 1.60934).alias("wind_speed_avg_kmh"),
    (df.wind_speed_max * 1.60934).alias("wind_speed_max_kmh"),
    (df.wind_speed_min * 1.60934).alias("wind_speed_min_kmh"),
    (df.wind_gust_avg * 1.60934).alias("wind_gust_avg_kmh"),
    (df.wind_gust_max * 1.60934).alias("wind_gust_max_kmh"),
    (df.wind_gust_min * 1.60934).alias("wind_gust_min_kmh"),
    (df.visibility_avg * 1.60934).alias("visibility_avg_km"),
    (df.visibility_max * 1.60934).alias("visibility_max_km"),
    (df.visibility_min * 1.60934).alias("visibility_min_km"),
    "*"
)



In [0]:
df_converted.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev_silver.accuweather.forecast_daily_calendar_metric")

In [0]:
from pyspark.sql.functions import col

display(df_converted.filter(col("city_name") == 'hong kong'))